# Async distillation: a sanity check on one example

**Qwen3-4B solves 17 × 23 → check the answer → train Qwen3-0.6B → run the student independently.**

Run the six cells from top to bottom using the repository's `.venv` kernel.
`sanity.py` contains process launching, training, and checks.
Generation uses the original `async_thoughts`; training uses the existing replay in `minisgl`.
Full logs and checkpoints are saved under `RUN`.


## 1. Settings

Select an idle GPU with at least 12000 MiB of free memory.
Teacher and student use the same token budgets and probe period. Each execution
of this cell creates a new directory and preserves previous results.


In [7]:
import json
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "scripts/async_distillation_sanity_check").is_dir())
sys.path.insert(0, str(ROOT))
from transformers import AutoTokenizer

from scripts.async_distillation_sanity_check.sanity import (
    check_answer,
    compare_rollouts,
    fit_student,
    load_example,
    record_rollout,
    verify_checkpoint,
)

GPU = "6"  # select an idle GPU
TEACHER = "Qwen/Qwen3-4B"
STUDENT = "Qwen/Qwen3-0.6B"
PROMPT = r"What is 17 * 23? Give one short calculation and put only the final integer in \boxed{...}."
EXPECTED = [17 * 23]  # independent reference: 391
LIMITS = {"think_tokens": 128, "answer_tokens": 192, "probe_period": 4}
STEPS, LR = 100, 1e-6

runs = ROOT / "training_runs/async_distillation_sanity_check"
runs.mkdir(parents=True, exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="simple_multiply_", dir=runs))
tokenizer = AutoTokenizer.from_pretrained(STUDENT)
print("Run directory:", RUN)


Run directory: /home/rozaevama/async_llm/training_runs/async_distillation_sanity_check/simple_multiply_fwvbrra4


## 2. Generate a correct teacher rollout

The 4B model generates thoughts and an answer through the original `async_thoughts` in BF16.
The trace records all requests, cache contexts, and sampled tokens. Independently check
the final boxed answer: it must be 391. An incorrect or truncated answer stops
execution **before student training**.


In [8]:
teacher_trace = record_rollout(ROOT, GPU, TEACHER, PROMPT, RUN / "teacher", **LIMITS)
teacher = load_example(teacher_trace)
print(tokenizer.decode(teacher["writer_tokens"], skip_special_tokens=True))

teacher_grade = check_answer(teacher, tokenizer, EXPECTED)
print("Teacher check:", teacher_grade["status"])
assert teacher_grade["status"] == "PASS", teacher_grade
print("Supervised targets:", sum(t != -100 for t in teacher["targets"]))


Generating teacher...
To calculate $ 17 \times 23 $, we can use the standard multiplication method:

$$
\begin{align*}
17 \\
\times 23 \\
\hline
51 & \quad \text{(17 × 3)} \\
34 & \quad \text{(17 × 20, shifted one position to the left)} \\
\hline
391 & \quad \text{(sum of 51 and 340)}
\end{align*}
$$

So, the final answer is:

$$
\boxed{391}
$$
Teacher check: PASS
Supervised targets: 258


## 3. Memorize one trajectory

Each optimizer step uses all supervised tokens in this example.
The student receives teacher tokens and the recorded context visibility (teacher forcing).
Loss is mean CrossEntropy over thinker/writer targets; prompts and probe requests have no direct loss.

Forward uses BF16 with sensitive operations in FP32. Master weights, their gradients,
and AdamW states remain FP32. Gradient checkpointing is enabled. The learning rate is constant;
evaluation runs every 5 updates. Overfit success: **accuracy = 100%, loss < 0.01**.


In [9]:
metrics = fit_student(
    ROOT, GPU, teacher_trace, STUDENT, RUN / "trained",
    steps=STEPS, lr=LR,
)
print("Loss:", metrics["initial"]["loss"], "→", metrics["final"]["loss"])
print("Accuracy:", metrics["initial"]["accuracy"], "→", metrics["final"]["accuracy"])
print("Updates:", metrics["completed_steps"])
assert metrics["overfit_achieved"], "Overfit not reached; see training.log and metrics.json"
checkpoint = Path(metrics["checkpoint"])


initial: {'loss': 0.5548288822174072, 'accuracy': 0.8255813953488372}
step 5: {'loss': 0.44649937748908997, 'accuracy': 0.8527131782945736}
step 10: {'loss': 0.25671395659446716, 'accuracy': 0.937984496124031}
step 15: {'loss': 0.15029339492321014, 'accuracy': 0.9767441860465116}
step 20: {'loss': 0.060162678360939026, 'accuracy': 0.9922480620155039}
step 25: {'loss': 0.030897043645381927, 'accuracy': 1.0}
step 30: {'loss': 0.00616011768579483, 'accuracy': 1.0}
Loss: 0.5548288822174072 → 0.00616011768579483
Accuracy: 0.8255813953488372 → 1.0
Updates: 30


## 4. Verify the saved weights

A fresh process loads the checkpoint from disk and repeats replay evaluation without updating weights.
This confirms that the checkpoint retains overfit. This is still teacher forcing, not free generation.


In [10]:
reloaded = verify_checkpoint(
    ROOT, GPU, teacher_trace, checkpoint, RUN / "reloaded.json",
)
assert reloaded["accuracy"] == 1.0 and reloaded["loss"] < 0.01, reloaded
assert abs(reloaded["loss"] - metrics["final"]["loss"]) < 1e-5
print("Saved checkpoint retains overfit:", reloaded["loss"], reloaded["accuracy"])


Saved checkpoint retains overfit: 0.00616011768579483 1.0


## 5. Let the student solve the task independently

Load the trained checkpoint in BF16. Pass the original prompt and the same token budgets.
The student generates its own tokens and a new schedule; the teacher trace is not passed to inference.
The orchestrator starts the thinker, and a probe using the same model permits or pauses the writer.
Probe decisions were not directly supervised during training.


In [11]:
student_trace = record_rollout(
    ROOT, GPU, checkpoint, PROMPT, RUN / "student", **LIMITS,
)
student = load_example(student_trace)
print("THINKER:\n", tokenizer.decode(student["thinker_tokens"], skip_special_tokens=True))
print("WRITER:\n", tokenizer.decode(student["writer_tokens"], skip_special_tokens=True))


Generating student...
THINKER:
 Okay, so I need to figure out what 17 multiplied by 23 is. Let me think. Hmm, I remember that multiplication can sometimes be broken down into smaller parts. Maybe I can use the distributive property here. Let me try that.

First, I know that 17 times 20 is easier to calculate. Let me do that. 17 times 20 is the same as 17 times 2 times 10. 17 times 2 is 34, and then times 10 is 340. So that's 340.
WRITER:
 To calculate $ 17 \times 23 $, we can use the standard multiplication method:

$$
\begin{align*}
17 \\
\times 23 \\
\hline
51 & \quad \text{(17 × 3)} \\
34 & \quad \text{(17 × 20, shifted one position to the left)} \\
\hline
391 & \quad \text{(sum of 51 and 340)}
\end{align*}
$$

So, the final answer is:

$$
\boxed{391}
$$


## 6. Three independent checks

- **Answer:** the boxed integer must be 391; the teacher is graded the same way.
- **Structural async behavior:** both streams run, the writer starts before the last
  thinker samples, and it reads a growing thinker context; the cache graph is valid and probe requests exist.
  This does not establish the optimality of individual `yes/no` decisions.
- **Exact rollout match:** tokens, targets, and cache history match after consistently
  renaming block IDs. Different schedules count as a mismatch even if the answers are identical.

Display every status, including `FAIL`. Full details are saved in `checks.json`.
Answer correctness, async behavior, and exact teacher reproduction are separate outcomes;
successful replay overfit does not guarantee successful independent generation.


In [12]:
report = compare_rollouts(teacher_trace, student_trace, tokenizer, EXPECTED, RUN / "checks.json")
labels = {
    "teacher_answer": "Teacher answer correctness",
    "student_answer": "1. Student answer correctness",
    "teacher_async": "Teacher async behavior",
    "student_async": "2. Student async behavior (structure)",
    "exact_recorded_rollout": "3. Exact rollout match",
}
for name, label in labels.items():
    print(f"{label}: {report[name]['status']}")
    if report[name]["status"] != "PASS":
        print("  " + report[name]["reason"])
        details = report[name]["details"]
        if "expected" in details:
            print("  Answer / reference:", details.get("actual"), "/", details["expected"])
        if details.get("failed"):
            print("  Failed checks:", ", ".join(details["failed"]))

for role, comparison in report["exact_recorded_rollout"]["details"]["streams"].items():
    print(f"{role}: equal={comparison['equal']}, common prefix={comparison['common_prefix_tokens']}, "
          f"tokens teacher/student={comparison['teacher_length']}/{comparison['student_length']}")
difference = report["exact_recorded_rollout"]["details"]["first_trace_difference"]
if difference:
    print("Trace mismatch:", json.dumps(difference, ensure_ascii=False))
print("Full report:", RUN / "checks.json")


Teacher answer correctness: PASS
1. Student answer correctness: PASS
Teacher async behavior: PASS
2. Student async behavior (structure): PASS
3. Exact rollout match: PASS
thinker: equal=True, common prefix=128, tokens teacher/student=128/128
writer: equal=True, common prefix=129, tokens teacher/student=129/129
Full report: /home/rozaevama/async_llm/training_runs/async_distillation_sanity_check/simple_multiply_fwvbrra4/checks.json
